# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [3]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [4]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [5]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "project-999ca28f-35e8-45ea-8b7"             # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'project-999ca28f-35e8-45ea-8b7' in 'us-central1'.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [6]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [7]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [8]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '980888c0-8550-46a3-aa26-376603475f1a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Embark on a relaxing and artsy day trip near Sunnyvale, California, designed to be both enriching and affordable. This itinerary combines serene natural beauty with world-class art, culminating in a budget-friendly meal.

Here’s your spontaneous day trip plan:

## Relaxing & Artsy Day Trip: Sunnyvale to Saratoga & Palo Alto

### Morning (10:00 AM - 1:00 PM): Serenity at Hakone Estate and Gardens

Start your day with a tranquil escape to the Hakone Estate and Gardens in Saratoga, one of the oldest residential-style Japanese gardens in the Western Hemisphere. These 18 acres of serene landscapes offer a peaceful retreat with koi ponds, a Zen garden, a moon bridge, and historic structures. It's a perfect place to

Embark on a relaxing and artsy day trip near Sunnyvale, California, designed to be both enriching and affordable. This itinerary combines serene natural beauty with world-class art, culminating in a budget-friendly meal.

Here’s your spontaneous day trip plan:

## Relaxing & Artsy Day Trip: Sunnyvale to Saratoga & Palo Alto

### Morning (10:00 AM - 1:00 PM): Serenity at Hakone Estate and Gardens

Start your day with a tranquil escape to the Hakone Estate and Gardens in Saratoga, one of the oldest residential-style Japanese gardens in the Western Hemisphere. These 18 acres of serene landscapes offer a peaceful retreat with koi ponds, a Zen garden, a moon bridge, and historic structures. It's a perfect place to relax and appreciate the artistry of traditional Japanese landscape design.

*   **Location:** 21000 Big Basin Way, Saratoga, CA 95070
*   **Operating Hours:** On weekdays, the gardens are open from 10:00 AM to 5:00 PM, with the last admission at 4:30 PM.
*   **Admission:** Adult admission is $12. Seniors (65+) and Youth (5-17) receive discounted rates, and children 4 and under are free.
*   **Budget Tip:** Bring your own packed lunch to enjoy at the designated picnic area, as food and beverages are not allowed in the main garden.

### Afternoon (2:30 PM - 5:30 PM): Art Immersion at Cantor Arts Center

After a relaxing morning, head to the Cantor Arts Center at Stanford University in Palo Alto. This premier arts destination features diverse collections spanning 5,000 years of art, including a renowned Rodin Sculpture Garden. Stroll through the galleries and the outdoor sculpture garden for a dose of artistic inspiration.

*   **Location:** 328 Lomita Dr, Stanford, CA 94305
*   **Operating Hours:** On Thursdays, the Cantor Arts Center is open from 11:00 AM to 8:00 PM.
*   **Admission:** Admission to the Cantor Arts Center is always free for everyone, including access to all permanent collection galleries and special exhibitions.
*   **Budget Tip:** Parking on the Stanford campus is enforced on weekdays between 8:00 AM and 4:00 PM and requires payment via the ParkMobile app. Plan for this potential cost or consider public transportation.

### Evening (6:00 PM - 7:30 PM): Affordable Dinner in Palo Alto

Conclude your day with an affordable and delicious dinner in Palo Alto. The area offers several budget-friendly options to satisfy your cravings.

*   **Suggestion:** Consider **Oren's Hummus** or **Zareen's** for flavorful and reasonably priced Middle Eastern or Pakistani/Indian cuisine. Other affordable options include **Mediterranean Wraps**, **Curry Up Now**, or **Taqueria El Grullense** for Mexican fare.
*   **Budget Tip:** Look for casual eateries that offer generous portions at lower price points to keep your day trip economical.

Enjoy your relaxing and artsy adventure!

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [9]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [10]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: '8d4f61a9-acf4-41a4-a512-b276fba2a175'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-a6827f99-fc84-47a9-ae03-54a9b57f11e7',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\n\xb9\x02\x01\x8f=k_\'\xda\xa4\xf9<\xb1\xba\xc8\xb6\xe8c~D\x11+\x03B\x9a\xd9\xca\x1f5\x13vl\x82\x99\x8b)\xaf\xbb>\x03\x93A\xf1\x06\xba\xa5\x16\xb5\x89v>\xa2\x16\x8f\x02\x8e\xc4\xe7\xd3\xc8\x95\x08\x9b\xae\x8a9D\xde\xb6\xd1\xea\xd6H)\xf6\xc8\x10/\xdd@\xadR\xd3\xb3V\n\xec\t\x12\n#d4"\xa9F...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=10
  

The weather near Lake Tahoe is 41°F and partly cloudy, with an east wind of 5 to 10 mph. It might be a bit chilly for a hike, so I recommend dressing in warm layers.

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [11]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [12]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: '1dc3c4b5-45b3-4363-8f59-66342cb093b1'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'top-rated hotels in San Francisco'
        },
        id='adk-7586e2df-818b-4f13-94e7-5da584ae806e',
        name='call_db_agent'
      ),
      thought_signature=b'\n\xa5\x04\x01\x8f=k_\xbd?\xae\xdf^\xf7\x1f\xc6`\x84r\x0b\x8e\xa9\xc9\x0c~\x17\x03\xb3"\x9d\xbc;\x10x\x83\xea\x13\xc3\x85F\x85\x17\xd5\xc5\xdeW\x82\xdf\xf6Bz[X\x11w\xa3Wqv\x17\x01\x84f\xa4\xcd\xaa\x04\xa3\x9b\x93\x1d\xca\x7f)\xf9\x98Z\xd0\xa7\xf2c{\xd6\xa5\x7f7\xbf}.\x019Q\xba\xc0\xb9\xf3\xaf...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None

Here are the top-rated hotels in San Francisco:

*   **The Grand Hotel**: 5-star rating, 450 reviews
*   **Seaside Inn**: 4-star rating, 620 reviews

The "Seaside Inn" has the most reviews. For a distinguished dinner spot near the Seaside Inn, I would recommend The Lobster Pot. It appears to be a notable choice for those with a discerning palate.

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [13]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [14]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 070e466d-8db5-4c56-be34-7b710f5d9b6d

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '070e466d-8db5-4c56-be34-7b710f5d9b6d'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Olá! Que ótimo! Lisboa é uma cidade fantástica para os seus interesses.

Vamos começar a planear o seu primeiro dia. Para o Dia 1 em Lisboa, considerando o seu interesse em locais históricos e excelente comida local, sugiro o seguinte:

**Dia 1: Descobrindo a História e os Sabores de Alfama e Baixa**

*   **Manhã (9:00 - 13:00): Bairro de Alfama e Castelo de São Jorge**
    *   Comece o seu dia no bairro mais antigo de Lisboa, Alfama. Passeie pelas suas ruas estreitas e labirínticas, admire as casas coloridas e sinta a atmosfera medieval.
   

Olá! Que ótimo! Lisboa é uma cidade fantástica para os seus interesses.

Vamos começar a planear o seu primeiro dia. Para o Dia 1 em Lisboa, considerando o seu interesse em locais históricos e excelente comida local, sugiro o seguinte:

**Dia 1: Descobrindo a História e os Sabores de Alfama e Baixa**

*   **Manhã (9:00 - 13:00): Bairro de Alfama e Castelo de São Jorge**
    *   Comece o seu dia no bairro mais antigo de Lisboa, Alfama. Passeie pelas suas ruas estreitas e labirínticas, admire as casas coloridas e sinta a atmosfera medieval.
    *   Suba até ao **Castelo de São Jorge**, uma das atrações mais emblemáticas de Lisboa. Explore as muralhas, as torres e desfrute das vistas panorâmicas deslumbrantes sobre a cidade e o rio Tejo.
*   **Almoço (13:00 - 14:30): Petiscos Tradicionais em Alfama**
    *   Desfrute de um almoço num dos pequenos restaurantes acolhedores em Alfama, onde poderá provar petiscos e pratos tradicionais portugueses.
*   **Tarde (14:30 - 18:00): Baixa Pombalina e Praça do Comércio**
    *   Desça para a **Baixa Pombalina**, o centro histórico reconstruído após o terramoto de 1755. Percorra as suas ruas ortogonais, admire a arquitetura neoclássica e explore as lojas tradicionais.
    *   Caminhe pela **Rua Augusta** até chegar à magnífica **Praça do Comércio**, uma das maiores e mais impressionantes praças da Europa, virada para o rio Tejo.
*   **Jantar (19:30): Sabores da Baixa**
    *   Sugiro jantar num restaurante na zona da Baixa ou Chiado, onde encontrará uma vasta oferta de restaurantes com gastronomia portuguesa, desde marisco fresco a pratos de carne.

O que acha deste plano para o seu primeiro dia?

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '070e466d-8db5-4c56-be34-7b710f5d9b6d'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Compreendido! Sem problemas, vamos ajustar o plano para o Dia 1. Entendo que castelos não sejam do seu agrado, então vamos trocar o Castelo de São Jorge por outro local com uma rica história.

Aqui está o plano revisado para o seu primeiro dia em Lisboa:

**Dia 1: Descobrindo a História e os Sabores de Alfama e Baixa**

*   **Manhã (9:00 - 13:00): Bairro de Alfama e Sé de Lisboa**
    *   Comece o seu dia no bairro mais antigo de Lisboa, Alfama. Passeie pelas suas ruas estreitas e labirínticas, admire as casas coloridas e sinta a atmosfera medieval.
    *   Em vez do castelo

Compreendido! Sem problemas, vamos ajustar o plano para o Dia 1. Entendo que castelos não sejam do seu agrado, então vamos trocar o Castelo de São Jorge por outro local com uma rica história.

Aqui está o plano revisado para o seu primeiro dia em Lisboa:

**Dia 1: Descobrindo a História e os Sabores de Alfama e Baixa**

*   **Manhã (9:00 - 13:00): Bairro de Alfama e Sé de Lisboa**
    *   Comece o seu dia no bairro mais antigo de Lisboa, Alfama. Passeie pelas suas ruas estreitas e labirínticas, admire as casas coloridas e sinta a atmosfera medieval.
    *   Em vez do castelo, visitaremos a **Sé de Lisboa (Catedral de Lisboa)**. Esta é a igreja mais antiga da cidade, com uma arquitetura imponente que mistura os estilos românico e gótico. Explore o seu interior, que conta séculos de história da cidade, e o claustro gótico. É um local histórico fascinante e muito central.
*   **Almoço (13:00 - 14:30): Petiscos Tradicionais em Alfama**
    *   Desfrute de um almoço num dos pequenos restaurantes acolhedores em Alfama, onde poderá provar petiscos e pratos tradicionais portugueses.
*   **Tarde (14:30 - 18:00): Baixa Pombalina e Praça do Comércio**
    *   Desça para a **Baixa Pombalina**, o centro histórico reconstruído após o terramoto de 1755. Percorra as suas ruas ortogonais, admire a arquitetura neoclássica e explore as lojas tradicionais.
    *   Caminhe pela **Rua Augusta** até chegar à magnífica **Praça do Comércio**, uma das maiores e mais impressionantes praças da Europa, virada para o rio Tejo.
*   **Jantar (19:30): Sabores da Baixa**
    *   Sugiro jantar num restaurante na zona da Baixa ou Chiado, onde encontrará uma vasta oferta de restaurantes com gastronomia portuguesa, desde marisco fresco a pratos de carne.

Como soa esta alternativa para a manhã do Dia 1?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '070e466d-8db5-4c56-be34-7b710f5d9b6d'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excelente! Fico feliz que o plano para o Dia 1 esteja do seu agrado. Vamos agora para o Dia 2, mantendo o foco em locais históricos e, claro, na maravilhosa gastronomia de Lisboa.

Para o seu **Dia 2** em Lisboa, sugiro um mergulho na época dos Descobrimentos e mais algumas experiências gastronómicas imperdíveis:

**Dia 2: A Época dos Descobrimentos e Experiências Gastronómicas Variadas**

*   **Manhã (9:00 - 13:00): Belém - O Legado dos Navegadores**
    *   Comece o dia com uma viagem até Belém, um bairro que pulsa com a história marítima de Portugal. Este é o local de onde partiram muitas das grandes expedições

Excelente! Fico feliz que o plano para o Dia 1 esteja do seu agrado. Vamos agora para o Dia 2, mantendo o foco em locais históricos e, claro, na maravilhosa gastronomia de Lisboa.

Para o seu **Dia 2** em Lisboa, sugiro um mergulho na época dos Descobrimentos e mais algumas experiências gastronómicas imperdíveis:

**Dia 2: A Época dos Descobrimentos e Experiências Gastronómicas Variadas**

*   **Manhã (9:00 - 13:00): Belém - O Legado dos Navegadores**
    *   Comece o dia com uma viagem até Belém, um bairro que pulsa com a história marítima de Portugal. Este é o local de onde partiram muitas das grandes expedições marítimas.
    *   Visite o **Mosteiro dos Jerónimos**, uma das joias arquitetónicas de Portugal e Património Mundial da UNESCO. Dedique tempo a explorar a igreja majestosa e os claustros ricamente decorados.
    *   Dali, caminhe até à icónica **Torre de Belém** e ao imponente **Padrão dos Descobrimentos**, ambos símbolos gloriosos da Era dos Descobrimentos.
*   **Almoço (13:00 - 14:30): Sabores e Doçuras de Belém**
    *   Em Belém, é obrigatório provar os originais e mundialmente famosos **Pastéis de Belém** na sua pastelaria original. Para o almoço, há vários restaurantes na área que servem pratos tradicionais portugueses.
*   **Tarde (14:30 - 18:00): Time Out Market (Mercado da Ribeira) e Elevador de Santa Justa**
    *   Regresse ao centro da cidade e dirija-se ao **Time Out Market (Mercado da Ribeira)**. Este espaço vibrante reúne alguns dos melhores restaurantes de Lisboa, bancas de comida tradicional e produtos frescos, permitindo-lhe provar uma enorme variedade de pratos e petiscos num único local. É uma verdadeira experiência gastronómica.
    *   Depois, pode subir no histórico **Elevador de Santa Justa** (se as filas não forem muito grandes) para desfrutar de vistas panorâmicas sobre a Baixa e o Castelo, ou simplesmente admirar a sua impressionante estrutura de ferro.
*   **Jantar (19:30): Jantar de Despedida com Fado (Opcional)**
    *   Para o seu jantar de despedida, sugiro uma experiência mais cultural. Explore os bairros do **Bairro Alto** ou **Alfama** (se não jantou lá no Dia 1) para encontrar um restaurante tradicional que ofereça **Fado ao vivo** enquanto desfruta de uma deliciosa refeição portuguesa. É uma forma autêntica de mergulhar na cultura e gastronomia locais.

O que acha desta proposta para o seu segundo e último dia em Lisboa?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [15]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 28321fbe-7b4b-4a38-b41e-425b0e7cd9ab
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '28321fbe-7b4b-4a38-b41e-425b0e7cd9ab'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! Lisbon is a fantastic choice with its rich history and delicious cuisine.

Let's start planning Day 1 of your trip. How about this itinerary:

### Day 1: Historic Alfama & Downtown Charm

*   **Morning (9:00 AM - 1:00 PM): Explore Castelo de São Jorge & Alfama.** Begin your day at the iconic Castelo de São Jorge, a historic Moorish castle offering incredible panoramic views of the city. Afterward, wander through the 

Great! Lisbon is a fantastic choice with its rich history and delicious cuisine.

Let's start planning Day 1 of your trip. How about this itinerary:

### Day 1: Historic Alfama & Downtown Charm

*   **Morning (9:00 AM - 1:00 PM): Explore Castelo de São Jorge & Alfama.** Begin your day at the iconic Castelo de São Jorge, a historic Moorish castle offering incredible panoramic views of the city. Afterward, wander through the labyrinthine streets of Alfama, Lisbon's oldest district, soaking in its medieval charm and discovering hidden viewpoints.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama.** Find a local *tasca* (tavern) in Alfama to savor authentic Portuguese dishes like Bacalhau à Brás or grilled sardines.
*   **Afternoon (2:30 PM - 6:00 PM): Discover Baixa and Chiado.** Head down to the elegant Baixa district, rebuilt after the 1755 earthquake, characterized by its grand squares like Praça do Comércio and Rua Augusta Arch. Then, stroll into the chic Chiado neighborhood, known for its historic theaters, traditional shops, and famous cafés like "A Brasileira."
*   **Evening (7:00 PM onwards): Dinner & Fado in Bairro Alto.** Conclude your day with a memorable dinner in Bairro Alto, a charming hilltop district that comes alive at night. Enjoy local cuisine at one of its many restaurants and, if you wish, experience a live Fado show – Portugal's soulful traditional music.

How does this sound for your first day in Lisbon?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: ef864621-b210-4b6c-aa37-6d0763848de5
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'ef864621-b210-4b6c-aa37-6d0763848de5'...

--------------------------------------------------
✅ Final Response:


An error occurred: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
